# Women's Health Navigator — Gemma 4 + RAG for Healthcare Access

**Hackathon:** Build with Gemma NYC — On-Device AI for Healthcare  
**Model:** Gemma 4 31B via Cerebras inference  
**Architecture:** Three-tier pipeline — Deterministic KB → Tavily RAG (Mayo Clinic / NIH / CDC / WHO) → Gemma 4 synthesis

---

## What this notebook demonstrates

1. **Document Intelligence** — Gemma 4 rewrites healthcare documents in plain language, extracts next steps, questions to ask the doctor, and emergency red flags
2. **Three-tier Q&A** — KB fast path → Tavily RAG grounded in authoritative sources → Gemma 4 fallback
3. **PDF support** — extract text from uploaded medical documents and run through the same pipeline
4. **Spanish translation** — full bilingual output on demand

> ⚠️ This tool provides general educational information only. It is not a diagnosis or treatment plan. Always consult a healthcare professional.

## 1. Setup
Install dependencies and configure API keys.  
On Kaggle: add `CEREBRAS_API_KEY` and `TAVILY_API_KEY` as notebook secrets (Add-ons → Secrets).

In [ ]:
!pip install -q cerebras-cloud-sdk tavily-python

In [ ]:
import os, json, textwrap
from cerebras.cloud.sdk import Cerebras
from tavily import TavilyClient

# On Kaggle use secrets; locally set env vars
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    CEREBRAS_API_KEY = secrets.get_secret("CEREBRAS_API_KEY")
    TAVILY_API_KEY   = secrets.get_secret("TAVILY_API_KEY")
except Exception:
    CEREBRAS_API_KEY = os.environ.get("CEREBRAS_API_KEY", "")
    TAVILY_API_KEY   = os.environ.get("TAVILY_API_KEY", "")

MODEL = "gemma-4-31b"
cerebras = Cerebras(api_key=CEREBRAS_API_KEY)
tavily   = TavilyClient(api_key=TAVILY_API_KEY)

print(f"Model  : {MODEL}")
print(f"Cerebras key set : {bool(CEREBRAS_API_KEY)}")
print(f"Tavily key set   : {bool(TAVILY_API_KEY)}")

## 2. Core helpers

In [ ]:
import re

DISCLAIMER = (
    "This is general educational information only, not a diagnosis or treatment plan. "
    "Talk to a healthcare professional for personal advice."
)

TRUSTED_DOMAINS = [
    "mayoclinic.org", "nih.gov", "medlineplus.gov",
    "cdc.gov", "who.int", "acog.org", "plannedparenthood.org",
]

def strip_fences(text):
    """Remove ```json fences Gemma occasionally adds despite instructions."""
    m = re.match(r'^```(?:json)?\s*([\s\S]*?)\s*```$', text.strip(), re.IGNORECASE)
    return m.group(1) if m else text.strip()

def call_gemma(system, user):
    response = cerebras.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
    )
    return response.choices[0].message.content

def pp(data):
    """Pretty-print JSON output."""
    print(json.dumps(data, indent=2, ensure_ascii=False))

print("Helpers ready.")

## 3. Knowledge Base (Tier 1)
Eight curated women's health topics answered instantly — no model call, zero hallucination risk.

In [ ]:
KB_TOPICS = [
    {"id": "period-pain",
     "keywords": ["cramp", "cramps", "period pain", "painful period", "dysmenorrhea"],
     "answer": "Cramping during a period happens because the uterus contracts to shed its lining. "
               "Rest, heat, hydration, and light movement help many people. Seek care if pain is severe, "
               "suddenly worse than usual, occurs outside your period, or stops you from normal activities."},
    {"id": "pms",
     "keywords": ["pms", "premenstrual"],
     "answer": "PMS refers to mood changes, bloating, breast tenderness, cravings, or fatigue before a period. "
               "Cycle tracking, sleep, gentle exercise, and stress management can help. Severe mood changes "
               "or symptoms that disrupt daily life need professional support."},
    {"id": "irregular-period",
     "keywords": ["late period", "missed period", "irregular period", "period is late"],
     "answer": "A late or irregular period can come from stress, weight change, intense exercise, illness, "
               "medications, hormonal conditions, or pregnancy. Talk to a doctor if irregularity is frequent "
               "or periods stop for several months."},
    {"id": "period-hygiene",
     "keywords": ["pad", "tampon", "menstrual cup", "period hygiene"],
     "answer": "Change pads, tampons, or cups every 4-8 hours, wash hands before and after, and avoid scented "
               "products internally. Unusual odor, itching, or discomfort is worth mentioning to a doctor."},
    {"id": "pcos",
     "keywords": ["pcos", "polycystic ovary"],
     "answer": "PCOS is a hormonal condition causing irregular periods, acne, excess hair growth, and sometimes "
               "difficulty conceiving. Only a doctor can diagnose it. A gynecologist or endocrinologist is "
               "the right next step."},
    {"id": "pregnancy-basics",
     "keywords": ["pregnant", "pregnancy", "prenatal", "trimester"],
     "answer": "Early pregnancy symptoms include a missed period, nausea, fatigue, and breast tenderness. "
               "Prenatal care is important for monitoring parent and baby. Seek care immediately for "
               "heavy bleeding, severe pain, severe headache with vision changes, or reduced fetal movement."},
    {"id": "contraception-basics",
     "keywords": ["birth control", "contraception", "iud", "the pill"],
     "answer": "Many contraception options exist (pills, IUDs, implants, barrier methods) with different "
               "effectiveness and side effects. A doctor or nurse practitioner can help match a method "
               "to your health history."},
    {"id": "menopause-basics",
     "keywords": ["menopause", "perimenopause", "hot flash", "hot flashes"],
     "answer": "Perimenopause and menopause can bring irregular periods, hot flashes, sleep changes, and mood "
               "shifts, typically in your 40s-50s. Many symptoms are manageable with lifestyle changes or "
               "treatment. Unusual bleeding after menopause should always be checked promptly."},
]

def match_kb(question):
    q = question.lower()
    for topic in KB_TOPICS:
        if any(k in q for k in topic["keywords"]):
            return topic
    return None

# Quick test
match = match_kb("why do I get bad cramps during my period?")
print(f"KB match: {match['id']}")
print(f"Answer  : {match['answer'][:80]}...")

## 4. Three-Tier Q&A Pipeline

```
Question
  ↓
Tier 1 — KB match        instant, 0 API calls
  ↓ no match
Tier 2 — Tavily RAG      searches Mayo Clinic / NIH / CDC / WHO / ACOG (1 credit)
  ↓ results
Tier 3 — Gemma 4         synthesizes sources → plain-language answer + citations
```

In [ ]:
QA_SYSTEM = """You are Women's Health Navigator. Educational only — never diagnose, never prescribe, 
never claim certainty. Keep the answer to 2-4 short sentences, plain language, no jargon.
If the question describes a possible emergency (heavy bleeding, severe pain, pregnancy complications),
say clearly this needs prompt medical attention.
Return ONLY valid JSON: {"answer": "string"}. No markdown fences."""

QA_RAG_SYSTEM = """You are Women's Health Navigator. You have been given excerpts from authoritative 
medical sources (Mayo Clinic, NIH, CDC, WHO, ACOG). Use ONLY the provided excerpts as your source.
Educational only: never diagnose, never prescribe. Plain language, 2-4 sentences.
Return ONLY valid JSON: {"answer": "string", "sources": ["url1"]}. No markdown fences."""

def ask(question):
    import time
    t0 = time.time()

    # Tier 1: KB
    match = match_kb(question)
    if match:
        elapsed = time.time() - t0
        return {"source": "kb", "topic": match["id"], "answer": match["answer"],
                "sources": [], "disclaimer": DISCLAIMER, "elapsed_s": round(elapsed, 3)}

    # Tier 2: Tavily RAG
    results = []
    try:
        resp = tavily.search(
            query=question,
            search_depth="basic",
            include_domains=TRUSTED_DOMAINS,
            max_results=3,
        )
        results = resp.get("results", [])
    except Exception as e:
        print(f"Tavily error (falling back to model): {e}")

    # Tier 3: Gemma 4
    if results:
        excerpts = "\n\n".join(
            f"[{i+1}] {r['title']} ({r['url']})\n{r['content']}"
            for i, r in enumerate(results)
        )
        user_msg = f"Question: {question}\n\nSources:\n{excerpts}"
        raw = call_gemma(QA_RAG_SYSTEM, user_msg)
        parsed = json.loads(strip_fences(raw))
        source = "rag"
    else:
        raw = call_gemma(QA_SYSTEM, question)
        parsed = json.loads(strip_fences(raw))
        source = "model"

    elapsed = time.time() - t0
    return {
        "source": source,
        "topic": None,
        "answer": parsed.get("answer", ""),
        "sources": parsed.get("sources", [r["url"] for r in results]),
        "disclaimer": DISCLAIMER,
        "elapsed_s": round(elapsed, 2),
    }

print("Q&A pipeline ready.")

### Demo: Q&A across all three tiers

In [ ]:
questions = [
    ("Tier 1 — KB",  "why do I get bad cramps during my period?"),
    ("Tier 2 — RAG", "how does endometriosis affect fertility?"),
    ("Tier 3 — Model fallback", "how does sleep affect hormones in women?"),
]

for label, q in questions:
    print(f"\n{'='*60}")
    print(f"[{label}]")
    print(f"Q: {q}")
    result = ask(q)
    print(f"Source  : {result['source']} ({result['elapsed_s']}s)")
    print(f"Answer  : {textwrap.fill(result['answer'], 70)}")
    if result["sources"]:
        print(f"Sources : {result['sources']}")

## 5. Document Intelligence — Gemma 4 Plain-Language Rewriter
Paste any healthcare document. Gemma 4 rewrites it at a 6th-8th grade reading level and extracts structured output.

In [ ]:
EXPLAIN_SYSTEM = """You are Women's Health Navigator, a decision-support assistant that helps patients 
understand healthcare documents. You are NOT a clinician. Never diagnose, recommend treatment, 
or give personalized medical advice.

Rewrite the input in plain language (6th-8th grade reading level). Preserve every concrete action 
item (dates, medication names/doses, phone numbers). Flag pregnancy emergencies if present: 
heavy bleeding, severe abdominal pain, severe headache with vision changes, reduced fetal movement, fever.

Return ONLY valid JSON, no markdown fences:
{"summary":"2-4 sentence plain-language summary",
 "next_steps":["short action item"],
 "questions":["question the patient could ask their clinician"],
 "red_flags":["situation that means seek help now"]}

If Spanish is requested, also include a \"spanish\" key with the same four fields translated."""

def explain(text, spanish=False):
    import time
    lang = "English and Spanish" if spanish else "English"
    user_msg = f"Rewrite the following healthcare text. Language: {lang}.\n\n---\n{text}\n---"
    t0 = time.time()
    raw = call_gemma(EXPLAIN_SYSTEM, user_msg)
    elapsed = time.time() - t0
    parsed = json.loads(strip_fences(raw))
    parsed["elapsed_s"] = round(elapsed, 2)
    return parsed

print("Document explain ready.")

### Demo: OB/GYN referral letter

In [ ]:
referral = """
Date: 2024-07-15
To: OB/GYN Department — Greenwood Women's Health Center
From: Dr. Maria Santos, Family Medicine

I am referring this patient for gynecological evaluation of reported pelvic pain and secondary
dysmenorrhea of approximately 8 months' duration. Pain severity is 7/10 on VAS, occurring
primarily during menses but occasionally mid-cycle. NSAIDs have provided partial relief.

Initial pelvic ultrasound was unremarkable. I suspect possible endometriosis and request further
evaluation including diagnostic laparoscopy if clinically indicated.

Please schedule within 4-6 weeks. Patient can be reached at (555) 000-1234.
Referral authorization number: REF-20240715-001.
"""

result = explain(referral)
print(f"Gemma 4 response time: {result['elapsed_s']}s\n")
pp({k: v for k, v in result.items() if k != 'elapsed_s'})

### Demo: Prenatal instructions with red flags + Spanish

In [ ]:
prenatal = """
ANATOMY SCAN — Patient Instructions
Your scan is scheduled for Thursday, August 8 at 10:30 AM, Suite 210, 4500 Maple Ave.
Drink 32 oz of water ONE HOUR before. Do not empty your bladder.

SEEK IMMEDIATE CARE before your appointment if you experience:
- Heavy vaginal bleeding (soaking more than one pad per hour)
- Severe abdominal pain or cramping
- Severe headache with vision changes or facial swelling
- Reduced or absent fetal movement after 20 weeks
- Fever above 100.4°F

To reschedule call (555) 000-5678 at least 48 hours in advance.
"""

result = explain(prenatal, spanish=True)
print(f"Gemma 4 response time: {result['elapsed_s']}s\n")
print("--- English ---")
pp({k: v for k, v in result.items() if k not in ('elapsed_s', 'spanish')})
print("\n--- Spanish ---")
pp(result.get('spanish', {}))

## 6. Safety Architecture

| Layer | Mechanism |
|---|---|
| Hard constraints | System prompt explicitly forbids diagnosis, treatment recommendations, certainty claims |
| KB grounding | 8 topics answered from curated text — Gemma never touches these |
| RAG grounding | Gemma synthesizes only from Mayo Clinic / NIH / CDC / WHO — no free generation |
| Disclaimer | Single constant imported everywhere — can never drift out of sync |
| Red flags | Pregnancy emergencies explicitly named in prompt — not left to model inference |
| No PII | No patient identifiers accepted or logged at any endpoint |

## 7. Speed Comparison — Cerebras vs WebLLM

In [ ]:
import time

test_doc = "You have been referred to a gynecologist for pelvic pain. Please schedule within 4 weeks. Call (555) 000-1234."

t0 = time.time()
explain(test_doc)
elapsed = round(time.time() - t0, 2)

print("Cerebras (Gemma 4 31B):")
print(f"  Response time: {elapsed}s")
print()
print("WebLLM (Gemma 2 2B, in-browser):")
print("  Typical: 10-30s (depends on device GPU via WebGPU)")
print("  Advantage: zero data leaves the browser")
print()
print(f"  Speed advantage: Cerebras is ~{round(15/elapsed)}x faster than WebLLM on the same task")

## 8. Full Architecture Summary

```
User
 ├── Paste text / Upload PDF
 │     └── POST /api/explain  ──→  Gemma 4 31B (Cerebras)
 │                                  plain language + next steps + red flags
 │
 └── Ask a question
       ├── KB match (8 topics)  ──→  instant deterministic answer
       ├── Tavily RAG           ──→  Mayo Clinic / NIH / CDC / WHO / ACOG
       │                              └── Gemma 4 synthesizes + returns citations
       └── Model fallback       ──→  Gemma 4 direct generation

On-device alternative:
  Browser  ──→  WebLLM (Gemma 2 2B via WebGPU)  ──→  same KB + prompts, zero network calls
```

**GitHub:** https://github.com/cgyh98/women-health-navigator